# Notebook 30: Capstone Project – Full End-to-End Kaggle-Style Solution + Deployment
**Part 30/30 – ML Mastery Series for Python Experts**

## Capstone Goal – From Raw Data to Live Predictions

This is it — the capstone where everything comes together. You've gone from your first model to a live, production-grade system. Let's build one complete, end-to-end project that showcases it all:

- **Realistic Kaggle workflow**: Complete data science pipeline from exploration to deployment
- **Combine pipelines + boosting + XAI + deployment**: Integrate all major concepts from the series
- **Emphasize reproducibility & production readiness**: Version control, deterministic results, proper testing
- **Showcase best practices learned**: Proper validation, no leakage, robust error handling
- **Evaluate on hold-out + cross-validation**: Rigorous model validation with nested CV
- **Interpret & debug with SHAP**: Explain predictions and identify model failures
- **Serve predictions via API & UI**: FastAPI for programmatic access, Streamlit for interactive demos
- **End-to-end automation**: Single command to go from raw data to deployed service

## Learning Objectives

By completing this capstone, you will demonstrate mastery of:

- Loading and exploring real-world datasets with proper train/validation/test splits
- Building full preprocessing pipelines with custom transformers and feature engineering
- Engineering advanced features safely without data leakage using sklearn pipelines
- Training and comparing boosting algorithms (XGBoost, LightGBM, CatBoost) or AutoML
- Tuning hyperparameters with early stopping, cross-validation, and Bayesian optimization
- Explaining model predictions with SHAP values, dependence plots, and partial dependence
- Serializing complete pipelines for production deployment with joblib
- Deploying models via FastAPI REST endpoints with input validation and error handling
- Building interactive Streamlit UIs for business users and stakeholders
- Testing live predictions and debugging production issues
- Reflecting on trade-offs and identifying improvement opportunities

## 🏆 1. Problem Setup & Data Loading

We'll use the Ames Housing dataset (House Prices) — a classic regression problem with mixed data types, missing values, and realistic feature engineering opportunities. This mirrors actual Kaggle competitions and real estate pricing models used in industry.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
%matplotlib inline
sns.set_theme(style="whitegrid", palette="husl")

# Load Ames Housing dataset from OpenML
print("Loading Ames Housing dataset...")
housing = fetch_openml(name="house_prices", version=1, as_frame=True, parser="auto")
X = housing.data
y = housing.target

print(f"Dataset shape: {X.shape}")
print(f"Target range: ${y.min():,.0f} to ${y.max():,.0f}")
print(f"Features: {len(X.columns)}")

# Display first few rows
X.head()

In [ ]:
# Quick data profiling
print("=" * 60)
print("DATA PROFILE")
print("=" * 60)

# Target distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw target
axes[0].hist(y, bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Sale Price ($)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Target Distribution (Raw)')

# Log-transformed target (often better for regression)
axes[1].hist(np.log1p(y), bins=50, edgecolor='black', alpha=0.7, color='green')
axes[1].set_xlabel('Log(Sale Price + 1)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Target Distribution (Log-transformed)')

plt.tight_layout()
plt.show()

# Check skewness
from scipy.stats import skew
print(f"\nTarget skewness: {skew(y):.2f} (log-transformed: {skew(np.log1p(y)):.2f})")
print("Note: We'll use log-transform for modeling to handle right skew.")

In [ ]:
# Missing values analysis
missing = X.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

print(f"Features with missing values: {len(missing)}")
print("\nTop 10 missing value counts:")
print(missing.head(10))

# Visualize missing values pattern
plt.figure(figsize=(12, 6))
sns.heatmap(X.isnull().iloc[:, :20], cbar=True, yticklabels=False, cmap='viridis')
plt.title('Missing Values Heatmap (First 20 Features)')
plt.xlabel('Features')
plt.show()

# Data types breakdown
print(f"\nData types:")
print(X.dtypes.value_counts())

In [ ]:
# Feature correlation with target (for numeric features only)
numeric_features = X.select_dtypes(include=[np.number]).columns
correlations = X[numeric_features].corrwith(pd.Series(y, index=X.index)).abs().sort_values(ascending=False)

plt.figure(figsize=(10, 6))
correlations.head(15).plot(kind='barh')
plt.title('Top 15 Numeric Features by Correlation with Sale Price')
plt.xlabel('Absolute Correlation')
plt.tight_layout()
plt.show()

print("Top correlations:")
print(correlations.head(10))

In [ ]:
# Train/validation/hold-out split
# Hold-out set is for final evaluation only — never seen during training or tuning
X_temp, X_hold, y_temp, y_hold = train_test_split(
    X, y, test_size=0.15, random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.176, random_state=42  # 0.176 * 0.85 ≈ 0.15 of total
)

print(f"Train set: {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Validation set: {X_val.shape[0]} samples ({X_val.shape[0]/len(X)*100:.1f}%)")
print(f"Hold-out test: {X_hold.shape[0]} samples ({X_hold.shape[0]/len(X)*100:.1f}%)")

# Apply log transform to target for training (reduces skewness)
y_train_log = np.log1p(y_train)
y_val_log = np.log1p(y_val)
y_hold_log = np.log1p(y_hold)

## 🔧 2. Full Preprocessing & Feature Engineering Pipeline

Production ML requires robust preprocessing that handles new data consistently. We'll build a comprehensive pipeline with:
- Custom transformers for domain-specific feature engineering
- Proper handling of categorical variables (including high-cardinality)
- Missing value imputation strategies tailored to feature types
- Feature selection integrated into the pipeline

All transformations happen inside CV folds to prevent data leakage.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PowerTransformer
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestRegressor
import category_encoders as ce

# Custom transformer: Create interaction features safely
class InteractionFeatures(BaseEstimator, TransformerMixin):
    """Create ratio and interaction features for housing data."""
    def __init__(self, interactions=None):
        self.interactions = interactions or [
            ('TotalBsmtSF', 'GrLivArea'),  # Basement to living ratio
            ('1stFlrSF', '2ndFlrSF'),       # Floor distribution
        ]
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X = X.copy()
        for feat1, feat2 in self.interactions:
            if feat1 in X.columns and feat2 in X.columns:
                # Avoid division by zero
                X[f'{feat1}_ratio_{feat2}'] = X[feat1] / (X[feat2] + 1)
                X[f'{feat1}_plus_{feat2}'] = X[feat1] + X[feat2]
        return X

# Custom transformer: Handle skewed numeric features
class SkewnessTransformer(BaseEstimator, TransformerMixin):
    """Apply log1p transform to highly skewed features."""
    def __init__(self, threshold=0.75):
        self.threshold = threshold
        self.skewed_features = []
    
    def fit(self, X, y=None):
        # X is DataFrame at this point
        if isinstance(X, pd.DataFrame):
            numeric = X.select_dtypes(include=[np.number])
            self.skewed_features = [
                col for col in numeric.columns 
                if abs(skew(numeric[col].dropna())) > self.threshold
            ]
        return self
    
    def transform(self, X):
        X = X.copy()
        for col in self.skewed_features:
            if col in X.columns:
                X[col] = np.log1p(X[col].clip(lower=0))
        return X

print("Custom transformers defined.")

In [ ]:
# Identify feature types automatically
numeric_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

# Separate high-cardinality categoricals for target encoding
high_cardinality = [col for col in categorical_features if X_train[col].nunique() > 10]
low_cardinality = [col for col in categorical_features if X_train[col].nunique() <= 10]

print(f"Numeric features: {len(numeric_features)}")
print(f"Categorical (low cardinality): {len(low_cardinality)}")
print(f"Categorical (high cardinality): {len(high_cardinality)}")

# Numeric preprocessing pipeline
numeric_transformer = Pipeline(steps=[
    ('imputer', KNNImputer(n_neighbors=5)),  # Sophisticated imputation for numerics
    ('skewness', SkewnessTransformer(threshold=0.75)),  # Handle skewness
    ('scaler', StandardScaler())  # Standardize for boosting algorithms
])

# Low cardinality categoricals: One-hot encoding
low_cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# High cardinality categoricals: Target encoding (with regularization)
high_cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),
    ('target_enc', ce.TargetEncoder(smoothing=0.3))
])

# Combine all preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('low_cat', low_cat_transformer, low_cardinality),
        ('high_cat', high_cat_transformer, high_cardinality)
    ],
    remainder='drop'  # Drop any columns not explicitly handled
)

print("Preprocessing pipeline configured.")

In [ ]:
# Full pipeline with feature engineering and selection
full_pipeline = Pipeline(steps=[
    ('interactions', InteractionFeatures()),  # Add domain-specific features
    ('preprocessor', preprocessor),           # Handle all feature types
    ('selector', SelectFromModel(            # Feature selection
        RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
        max_features=100,                    # Keep top 100 features
        threshold=-np.inf                    # Select by ranking, not threshold
    ))
])

# Fit on training data to verify
print("Fitting pipeline on training data...")
X_train_processed = full_pipeline.fit_transform(X_train, y_train_log)
print(f"Original features: {X_train.shape[1]}")
print(f"After preprocessing: {X_train_processed.shape[1]}")
print(f"Sample shape check passed: {X_train_processed.shape}")

## 🚀 3. Model Selection & Training – Boosting / AutoML

We'll compare three state-of-the-art gradient boosting frameworks. Each has strengths:
- **XGBoost**: Battle-tested, excellent regularization, widely supported
- **LightGBM**: Fast training, handles large data efficiently, leaf-wise growth
- **CatBoost**: Superior handling of categorical features, reduces overfitting

We'll use early stopping on the validation set to prevent overfitting and optimize training time.

In [ ]:
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Prepare data for all models
X_train_proc = full_pipeline.fit_transform(X_train, y_train_log)
X_val_proc = full_pipeline.transform(X_val)
X_hold_proc = full_pipeline.transform(X_hold)

print(f"Processed shapes - Train: {X_train_proc.shape}, Val: {X_val_proc.shape}")

# Define models with early stopping
models = {
    'XGBoost': xgb.XGBRegressor(
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        early_stopping_rounds=50,
        eval_metric='rmse'
    ),
    'LightGBM': lgb.LGBMRegressor(
        n_estimators=1000,
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        early_stopping_rounds=50
    ),
    'CatBoost': CatBoostRegressor(
        iterations=1000,
        learning_rate=0.05,
        depth=6,
        random_seed=42,
        verbose=False,
        early_stopping_rounds=50
    )
}

results = {}

In [ ]:
# Train and evaluate each model
for name, model in models.items():
    print(f"\nTraining {name}...")
    
    if name == 'CatBoost':
        # CatBoost has different early stopping API
        model.fit(
            X_train_proc, y_train_log,
            eval_set=(X_val_proc, y_val_log),
            verbose=False
        )
    else:
        model.fit(
            X_train_proc, y_train_log,
            eval_set=[(X_val_proc, y_val_log)],
            verbose=False
        )
    
    # Predictions (convert back from log scale)
    val_pred_log = model.predict(X_val_proc)
    val_pred = np.expm1(val_pred_log)  # Inverse of log1p
    
    # Metrics on original scale
    rmse = np.sqrt(mean_squared_error(y_val, val_pred))
    mae = mean_absolute_error(y_val, val_pred)
    r2 = r2_score(y_val, val_pred)
    
    results[name] = {
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2,
        'model': model,
        'best_iter': model.best_iteration if hasattr(model, 'best_iteration') else model.get_best_iteration()
    }
    
    print(f"  RMSE: ${rmse:,.0f}")
    print(f"  MAE:  ${mae:,.0f}")
    print(f"  R²:   {r2:.4f}")
    if results[name]['best_iter']:
        print(f"  Best iteration: {results[name]['best_iter']}")

In [ ]:
# Compare models
comparison_df = pd.DataFrame({
    name: {k: v for k, v in metrics.items() if k != 'model'}
    for name, metrics in results.items()
}).T

print("=" * 60)
print("MODEL COMPARISON (Validation Set)")
print("=" * 60)
print(comparison_df[['RMSE', 'MAE', 'R2']].sort_values('RMSE'))

# Select best model
best_model_name = comparison_df['RMSE'].idxmin()
best_model = results[best_model_name]['model']
print(f"\n🏆 Best model: {best_model_name}")

# Visualize predictions vs actual
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, (name, model_info) in enumerate(results.items()):
    model = model_info['model']
    pred_log = model.predict(X_val_proc)
    pred = np.expm1(pred_log)
    
    axes[idx].scatter(y_val, pred, alpha=0.3, s=10)
    axes[idx].plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], 'r--', lw=2)
    axes[idx].set_xlabel('Actual Price')
    axes[idx].set_ylabel('Predicted Price')
    axes[idx].set_title(f'{name}\nRMSE: ${model_info["RMSE"]:,.0f}')
    axes[idx].ticklabel_format(style='plain')

plt.tight_layout()
plt.show()

In [ ]:
# Final training on combined train+validation with best model
print(f"Retraining {best_model_name} on full training set...")

X_full_train = pd.concat([X_train, X_val])
y_full_train_log = np.concatenate([y_train_log, y_val_log])

# Refit pipeline on full data
X_full_processed = full_pipeline.fit_transform(X_full_train, y_full_train_log)

# Train final model with slightly fewer iterations (already tuned)
if best_model_name == 'CatBoost':
    final_model = CatBoostRegressor(
        iterations=int(results[best_model_name]['best_iter'] * 1.1),  # 10% buffer
        learning_rate=0.05,
        depth=6,
        random_seed=42,
        verbose=False
    )
else:
    final_model = type(best_model)(
        n_estimators=int(results[best_model_name]['best_iter'] * 1.1),
        learning_rate=0.05,
        random_state=42,
        n_jobs=-1,
        **{k: v for k, v in best_model.get_params().items() 
           if k not in ['n_estimators', 'learning_rate', 'random_state', 'n_jobs', 'early_stopping_rounds']}
    )

final_model.fit(X_full_processed, y_full_train_log)
print("Final model trained.")

## 🔍 4. Evaluation & Interpretability

Before deployment, we must understand:
1. **Performance**: How well does it generalize to unseen hold-out data?
2. **Interpretability**: Which features drive predictions? Are relationships sensible?
3. **Debugging**: Where does the model fail? Systematic errors indicate feature gaps.

In [ ]:
# Final evaluation on hold-out test set (never seen during any training/tuning)
X_hold_processed = full_pipeline.transform(X_hold)
hold_pred_log = final_model.predict(X_hold_processed)
hold_pred = np.expm1(hold_pred_log)

# Calculate metrics
hold_rmse = np.sqrt(mean_squared_error(y_hold, hold_pred))
hold_mae = mean_absolute_error(y_hold, hold_pred)
hold_r2 = r2_score(y_hold, hold_pred)

print("=" * 60)
print("FINAL HOLD-OUT TEST RESULTS")
print("=" * 60)
print(f"RMSE: ${hold_rmse:,.0f}")
print(f"MAE:  ${hold_mae:,.0f}")
print(f"R²:   {hold_r2:.4f}")
print(f"Mean house price: ${y_hold.mean():,.0f}")
print(f"RMSE as % of mean: {hold_rmse/y_hold.mean()*100:.1f}%")

# Residual analysis
residuals = y_hold - hold_pred
relative_error = np.abs(residuals) / y_hold

print(f"\nMedian absolute percentage error: {np.median(relative_error)*100:.1f}%")
print(f"Percentage of predictions within 10%: {(relative_error < 0.10).mean()*100:.1f}%")
print(f"Percentage of predictions within 20%: {(relative_error < 0.20).mean()*100:.1f}%")

In [ ]:
# Residual plots for debugging
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Residuals vs predicted
axes[0, 0].scatter(hold_pred, residuals, alpha=0.4, s=10)
axes[0, 0].axhline(y=0, color='r', linestyle='--')
axes[0, 0].set_xlabel('Predicted Price')
axes[0, 0].set_ylabel('Residuals')
axes[0, 0].set_title('Residuals vs Predicted')

# Residuals vs actual
axes[0, 1].scatter(y_hold, residuals, alpha=0.4, s=10, color='green')
axes[0, 1].axhline(y=0, color='r', linestyle='--')
axes[0, 1].set_xlabel('Actual Price')
axes[0, 1].set_ylabel('Residuals')
axes[0, 1].set_title('Residuals vs Actual')

# Distribution of residuals
axes[1, 0].hist(residuals, bins=50, edgecolor='black', alpha=0.7)
axes[1, 0].set_xlabel('Residual ($)')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Distribution of Residuals')

# Actual vs Predicted
axes[1, 1].scatter(y_hold, hold_pred, alpha=0.4, s=10)
axes[1, 1].plot([y_hold.min(), y_hold.max()], [y_hold.min(), y_hold.max()], 'r--', lw=2)
axes[1, 1].set_xlabel('Actual Price')
axes[1, 1].set_ylabel('Predicted Price')
axes[1, 1].set_title('Actual vs Predicted (Hold-out)')

plt.tight_layout()
plt.show()

# Identify worst predictions
worst_idx = np.argsort(np.abs(residuals))[-5:]
print("\nWorst predictions (investigate these):")
for idx in worst_idx:
    actual = y_hold.iloc[idx] if hasattr(y_hold, 'iloc') else y_hold[idx]
    pred = hold_pred[idx]
    print(f"  Actual: ${actual:,.0f}, Predicted: ${pred:,.0f}, Error: ${abs(actual-pred):,.0f}")

In [ ]:
# SHAP interpretability
import shap

print("Computing SHAP values for interpretability...")

# Create explainer
explainer = shap.TreeExplainer(final_model)

# Sample for speed (SHAP can be slow on large datasets)
sample_size = min(500, len(X_hold_processed))
sample_idx = np.random.choice(len(X_hold_processed), sample_size, replace=False)
X_sample = X_hold_processed[sample_idx]

shap_values = explainer.shap_values(X_sample)

# Summary plot
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_sample, show=False, max_display=15)
plt.title('SHAP Feature Importance (Top 15)')
plt.tight_layout()
plt.show()

print("SHAP summary shows:")
print("- Feature importance (mean absolute SHAP value)")
print("- Direction of impact (color: red=high, blue=low feature value)")
print("- Distribution of impacts across samples")

In [ ]:
# Partial Dependence Plots for top features
from sklearn.inspection import partial_dependence, PartialDependenceDisplay

# Get feature names after preprocessing (approximate)
# In production, you'd track feature names through the pipeline
feature_names = [f'f{i}' for i in range(X_hold_processed.shape[1])]

# Select top 4 features by importance for PDP
top_features_idx = np.argsort(np.abs(shap_values).mean(0))[-4:]

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.ravel()

for idx, feat_idx in enumerate(top_features_idx):
    # Calculate PDP
    pd_result = partial_dependence(
        final_model, X_hold_processed, features=[feat_idx],
        kind='average', grid_resolution=20
    )
    
    axes[idx].plot(pd_result['grid_values'][0], pd_result['average'][0])
    axes[idx].set_xlabel(f'Feature {feat_idx}')
    axes[idx].set_ylabel('Partial Dependence')
    axes[idx].set_title(f'PDP for Feature {feat_idx}')

plt.suptitle('Partial Dependence Plots (Top 4 Features)', fontsize=14)
plt.tight_layout()
plt.show()

print("PDPs show marginal effect of features on predictions.")
print("Check for monotonicity and unexpected non-linearities.")

## 💾 5. Serialization & Production Pipeline

For deployment, we need to save:
1. The complete preprocessing pipeline (fitted)
2. The trained model
3. Metadata (version, training date, performance metrics)

We'll use joblib for efficient serialization of sklearn objects.

In [ ]:
import joblib
import json
from datetime import datetime

# Create production artifact
production_artifact = {
    'pipeline': full_pipeline,           # Complete preprocessing
    'model': final_model,                # Trained booster
    'model_type': best_model_name,       # Which algorithm won
    'metrics': {
        'holdout_rmse': hold_rmse,
        'holdout_mae': hold_mae,
        'holdout_r2': hold_r2,
        'median_ape': float(np.median(relative_error))
    },
    'metadata': {
        'version': '1.0.0',
        'created_at': datetime.now().isoformat(),
        'training_samples': len(X_full_train),
        'features_input': X_train.shape[1],
        'features_selected': X_full_processed.shape[1]
    }
}

# Save to disk
model_path = 'capstone_model_v1.joblib'
joblib.dump(production_artifact, model_path, compress=3)

print(f"✅ Model saved to: {model_path}")
print(f"   File size: {np.round(os.path.getsize(model_path) / 1024 / 1024, 2)} MB")

# Save metadata separately as JSON for easy inspection
metadata_path = 'capstone_model_metadata.json'
with open(metadata_path, 'w') as f:
    json.dump(production_artifact['metadata'], f, indent=2)
print(f"   Metadata saved to: {metadata_path}")

In [ ]:
# Verify loading and prediction consistency
print("\nVerifying saved model...")

# Load artifact
loaded_artifact = joblib.load(model_path)
loaded_pipeline = loaded_artifact['pipeline']
loaded_model = loaded_artifact['model']

# Test prediction on hold-out
X_hold_loaded = loaded_pipeline.transform(X_hold)
loaded_pred_log = loaded_model.predict(X_hold_loaded)
loaded_pred = np.expm1(loaded_pred_log)

# Verify identical predictions
max_diff = np.max(np.abs(loaded_pred - hold_pred))
print(f"Max prediction difference: {max_diff:.10f}")
assert max_diff < 1e-6, "Model loading failed - predictions don't match!"
print("✅ Loaded model produces identical predictions")

# Show model parameters available for tuning
print(f"\nModel parameters available for future tuning:")
params = loaded_model.get_params()
for key in list(params.keys())[:10]:  # Show first 10
    print(f"  {key}: {params[key]}")
print(f"  ... and {len(params)-10} more parameters")

## 🌐 6. FastAPI Deployment – REST Prediction Service

FastAPI provides high-performance async REST APIs with automatic validation and documentation. We'll create a prediction service that:
- Validates input data using Pydantic models
- Returns predictions with confidence intervals (where applicable)
- Includes health checks and model metadata endpoints
- Optionally returns SHAP values for explainability

In [ ]:
# Create FastAPI application code as string (for demonstration)
# In production, this would be in a separate file: api.py

fastapi_code = '''
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
import numpy as np
import joblib
from typing import List, Optional
import uvicorn

# Initialize app
app = FastAPI(
    title="House Price Prediction API",
    description="Production-grade house price prediction service",
    version="1.0.0"
)

# Load model at startup
model_artifact = joblib.load("capstone_model_v1.joblib")
pipeline = model_artifact['pipeline']
model = model_artifact['model']

# Define input schema based on Ames Housing features
class HouseFeatures(BaseModel):
    # Key numeric features (subset for demo)
    OverallQual: int = Field(..., ge=1, le=10, description="Overall material and finish quality")
    GrLivArea: float = Field(..., gt=0, description="Above grade living area square feet")
    TotalBsmtSF: float = Field(..., ge=0, description="Total basement square feet")
    GarageCars: float = Field(..., ge=0, description="Garage car capacity")
    YearBuilt: int = Field(..., ge=1872, le=2024, description="Original construction year")
    
    # Categorical features
    Neighborhood: str = Field(..., description="Physical locations within Ames city limits")
    HouseStyle: str = Field(..., description="Style of dwelling")
    
    # Optional additional features
    FullBath: Optional[float] = Field(2, ge=0, description="Full bathrooms above grade")
    BedroomAbvGr: Optional[float] = Field(3, ge=0, description="Bedrooms above grade")

class PredictionResponse(BaseModel):
    predicted_price: float
    price_log_scale: float
    model_version: str
    confidence: Optional[str] = None

@app.get("/")
def root():
    return {
        "message": "House Price Prediction API",
        "model_type": model_artifact['model_type'],
        "version": model_artifact['metadata']['version']
    }

@app.get("/health")
def health_check():
    return {"status": "healthy", "model_loaded": model is not None}

@app.get("/metadata")
def get_metadata():
    return model_artifact['metadata']

@app.post("/predict", response_model=PredictionResponse)
def predict(house: HouseFeatures):
    try:
        # Convert to DataFrame (single row)
        import pandas as pd
        input_df = pd.DataFrame([house.dict()])
        
        # Preprocess
        X_processed = pipeline.transform(input_df)
        
        # Predict (returns log scale)
        pred_log = float(model.predict(X_processed)[0])
        pred_price = float(np.expm1(pred_log))
        
        return PredictionResponse(
            predicted_price=round(pred_price, 2),
            price_log_scale=round(pred_log, 4),
            model_version=model_artifact['metadata']['version'],
            confidence="based_on_holdout_r2_" + str(round(model_artifact['metrics']['holdout_r2'], 2))
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/predict_batch")
def predict_batch(houses: List[HouseFeatures]):
    """Batch prediction endpoint for multiple houses."""
    try:
        import pandas as pd
        input_df = pd.DataFrame([h.dict() for h in houses])
        X_processed = pipeline.transform(input_df)
        preds_log = model.predict(X_processed)
        preds = np.expm1(preds_log)
        
        return {
            "predictions": [round(float(p), 2) for p in preds],
            "count": len(preds)
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)
'''

# Save API code
with open('api.py', 'w') as f:
    f.write(fastapi_code)

print("✅ FastAPI application saved to: api.py")
print("\nTo run the API:")
print("   uvicorn api:app --reload")
print("\nThen visit:")
print("   http://localhost:8000/docs  (Interactive API documentation)")
print("   http://localhost:8000/      (Health check)")

In [ ]:
# Demonstrate API usage with test request
print("\nSimulating API request...")

# Mock request data
test_house = {
    "OverallQual": 7,
    "GrLivArea": 1710,
    "TotalBsmtSF": 856,
    "GarageCars": 2,
    "YearBuilt": 2003,
    "Neighborhood": "CollgCr",
    "HouseStyle": "2Story",
    "FullBath": 2,
    "BedroomAbvGr": 3
}

# Simulate what API does
test_df = pd.DataFrame([test_house])
X_test_proc = full_pipeline.transform(test_df)
pred_log = final_model.predict(X_test_proc)[0]
pred_price = np.expm1(pred_log)

print(f"Input house: {test_house['Neighborhood']}, {test_house['GrLivArea']} sqft, built {test_house['YearBuilt']}")
print(f"Predicted price: ${pred_price:,.2f}")
print(f"\nAPI would return JSON:")
import json
print(json.dumps({
    "predicted_price": round(pred_price, 2),
    "price_log_scale": round(pred_log, 4),
    "model_version": "1.0.0"
}, indent=2))

## 🎨 7. Streamlit UI – Interactive Demo App

Streamlit enables rapid creation of data apps for business users. We'll build an interactive interface that:
- Provides intuitive sliders and inputs for house features
- Displays predictions with visual feedback
- Shows SHAP explanations for individual predictions
- Supports batch upload for multiple predictions

In [ ]:
# Create Streamlit application code
streamlit_code = '''
import streamlit as st
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt

# Page config
st.set_page_config(
    page_title="House Price Predictor",
    page_icon="🏠",
    layout="wide"
)

# Load model (cached for performance)
@st.cache_resource
def load_model():
    artifact = joblib.load("capstone_model_v1.joblib")
    return artifact['pipeline'], artifact['model'], artifact['metadata']

pipeline, model, metadata = load_model()

# Title
st.title("🏠 House Price Prediction App")
st.markdown("Enter house details to get an instant price estimate.")

# Sidebar for inputs
st.sidebar.header("House Features")

# Numeric inputs
overall_qual = st.sidebar.slider("Overall Quality (1-10)", 1, 10, 7)
gr_liv_area = st.sidebar.number_input("Living Area (sq ft)", min_value=300, max_value=6000, value=1500)
total_bsmt = st.sidebar.number_input("Basement Area (sq ft)", min_value=0, max_value=3000, value=800)
garage_cars = st.sidebar.slider("Garage Capacity (cars)", 0, 5, 2)
year_built = st.sidebar.slider("Year Built", 1870, 2024, 2000)
full_bath = st.sidebar.slider("Full Bathrooms", 0, 5, 2)
bedrooms = st.sidebar.slider("Bedrooms", 0, 10, 3)

# Categorical inputs
neighborhood = st.sidebar.selectbox(
    "Neighborhood",
    ["CollgCr", "Veenker", "Crawfor", "NoRidge", "Mitchel", 
     "Somerst", "NWAmes", "OldTown", "BrkSide", "Sawyer"]
)
house_style = st.sidebar.selectbox(
    "House Style",
    ["1Story", "2Story", "1.5Fin", "SLvl", "SFoyer", "2.5Unf"]
)

# Prediction button
if st.sidebar.button("Predict Price", type="primary"):
    # Create input DataFrame
    input_data = pd.DataFrame([{
        "OverallQual": overall_qual,
        "GrLivArea": gr_liv_area,
        "TotalBsmtSF": total_bsmt,
        "GarageCars": garage_cars,
        "YearBuilt": year_built,
        "FullBath": full_bath,
        "BedroomAbvGr": bedrooms,
        "Neighborhood": neighborhood,
        "HouseStyle": house_style
    }])
    
    # Predict
    X_proc = pipeline.transform(input_data)
    pred_log = model.predict(X_proc)[0]
    pred_price = np.expm1(pred_log)
    
    # Display results
    col1, col2, col3 = st.columns(3)
    
    with col1:
        st.metric("Predicted Price", f"${pred_price:,.0f}")
    
    with col2:
        # Price per sqft
        price_per_sqft = pred_price / gr_liv_area
        st.metric("Price per Sq Ft", f"${price_per_sqft:,.0f}")
    
    with col3:
        st.metric("Model Confidence", f"R² = {metadata.get('metrics', {}).get('holdout_r2', 'N/A'):.2f}")
    
    # Visualization
    st.subheader("Price Context")
    
    # Show where this falls in distribution
    fig, ax = plt.subplots()
    # Simulated distribution (in production, use actual training data distribution)
    np.random.seed(42)
    sample_prices = np.random.lognormal(12, 0.4, 1000)
    ax.hist(sample_prices, bins=50, alpha=0.7, color='skyblue', edgecolor='black')
    ax.axvline(pred_price, color='red', linestyle='--', linewidth=2, label='Your House')
    ax.set_xlabel('Price ($)')
    ax.set_ylabel('Frequency')
    ax.legend()
    st.pyplot(fig)
    
    # Feature importance for this prediction (simplified)
    st.subheader("Key Value Drivers")
    st.write(f"- **Quality Rating**: {overall_qual}/10 {'✅ Premium' if overall_qual >= 8 else '⚠️ Average' if overall_qual >= 5 else '❌ Below Average'}")
    st.write(f"- **Living Space**: {gr_liv_area:,} sq ft")
    st.write(f"- **Age**: {2024 - year_built} years old")
    st.write(f"- **Location**: {neighborhood}")

# Batch prediction section
st.markdown("---")
st.subheader("Batch Prediction")
uploaded_file = st.file_uploader("Upload CSV with house features", type=['csv'])

if uploaded_file:
    batch_df = pd.read_csv(uploaded_file)
    st.write(f"Uploaded {len(batch_df)} houses")
    st.dataframe(batch_df.head())
    
    if st.button("Run Batch Prediction"):
        X_batch = pipeline.transform(batch_df)
        preds_log = model.predict(X_batch)
        preds = np.expm1(preds_log)
        
        batch_df['Predicted_Price'] = preds
        st.success("Predictions complete!")
        st.dataframe(batch_df[['Predicted_Price']].describe())
        
        # Download link
        csv = batch_df.to_csv(index=False)
        st.download_button(
            label="Download Results",
            data=csv,
            file_name="predictions.csv",
            mime="text/csv"
        )

# Footer
st.markdown("---")
st.caption(f"Model version {metadata['version']} | Trained on {metadata['training_samples']} samples")
'''

with open('app.py', 'w') as f:
    f.write(streamlit_code)

print("✅ Streamlit application saved to: app.py")
print("\nTo run the UI:")
print("   streamlit run app.py")
print("\nFeatures:")
print("   - Interactive feature inputs with validation")
print("   - Real-time price prediction")
print("   - Price distribution visualization")
print("   - Batch prediction via CSV upload")
print("   - Model metadata display")

In [ ]:
# Display deployment architecture summary
print("\n" + "=" * 60)
print("DEPLOYMENT ARCHITECTURE")
print("=" * 60)

architecture = """
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│   Data Sources  │────▶│  Training Pipeline│────▶│  Serialized     │
│   (CSV/Database)│     │  (This Notebook)  │     │  Model (.joblib)│
└─────────────────┘     └──────────────────┘     └─────────────────┘
                                                          │
           ┌──────────────────────────────────────────────┘
           │
           ▼
┌─────────────────────────────────────────────────────────────────┐
│                     Production Environment                       │
│  ┌──────────────┐              ┌──────────────┐                │
│  │  FastAPI     │              │  Streamlit   │                │
│  │  (/predict)  │◀────────────▶│  (UI Demo)   │                │
│  │  REST API    │   API Calls  │  Interactive │                │
│  └──────────────┘              └──────────────┘                │
│         │                                                        │
│         ▼                                                        │
│  ┌──────────────┐                                                │
│  │   Model      │                                                │
│  │   Inference  │                                                │
│  └──────────────┘                                                │
└─────────────────────────────────────────────────────────────────┘
"""
print(architecture)

print("\nDeployment Checklist:")
print("  ✅ Model serialized with joblib")
print("  ✅ FastAPI service defined (api.py)")
print("  ✅ Streamlit UI defined (app.py)")
print("  ✅ Input validation via Pydantic")
print("  ✅ Batch prediction support")
print("  ✅ Health check endpoint")
print("  ⚠️  TODO: Add authentication for production")
print("  ⚠️  TODO: Add request logging")
print("  ⚠️  TODO: Set up monitoring/alerting")
print("  ⚠️  TODO: Containerize with Docker")

## ⚠️ Common Pitfalls & Pro Tips

Avoid these production ML mistakes:

- **🚰 Data leakage in feature engineering**: Never calculate statistics (means, encodings) on the full dataset before splitting. Always fit transformers on training data only.

- **📊 Poor hold-out practices**: If you peek at the hold-out set during model selection, it's not a true hold-out. Use nested cross-validation for unbiased estimates.

- **🔌 Not validating API inputs**: Always use Pydantic models with constraints (min/max values) to prevent crashes from malformed requests.

- **🐢 Streamlit caching issues**: Use `@st.cache_resource` for model loading, not `@st.cache_data`. Models aren't hashable and shouldn't be reloaded on every interaction.

- **⏱️ Large SHAP compute in UI**: Calculating SHAP values for every prediction in the UI causes timeouts. Precompute or sample, or make it an async background task.

- **🔢 Version mismatch between dev & prod**: Pin all dependency versions (requirements.txt) to prevent "works on my machine" issues with sklearn/xgboost versions.

- **🐌 Ignoring latency in batch predict**: For large batches, implement chunked processing or background jobs (Celery/RQ) rather than blocking the API.

- **📝 Not logging API requests**: Log inputs, outputs, and prediction times for debugging and audit trails. Essential for production monitoring.

- **🧪 Insufficient testing**: Test the full pipeline with edge cases (zero values, missing fields, extreme outliers) before deployment.

- **🔄 Not versioning data**: Model performance depends on training data. Version your datasets (DVC) alongside model versions for reproducibility.

- **🛡️ No fallback for model failures**: Implement circuit breakers and default responses if the model service goes down.

- **📈 Forgetting to retrain**: Models drift. Set up automated retraining pipelines when performance degrades or data accumulates.

## 📝 Exercises & Extensions

Take this capstone further with these challenges:

### Easy
**Exercise 1**: Swap the dataset to Titanic (classification) and re-run the full workflow. Adjust preprocessing for the new data types (mixed categorical/numeric) and change metrics to AUC/F1.

### Medium
**Exercise 2**: Integrate SHAP force plots into the Streamlit UI. When a user clicks "Predict", show a force plot explaining which features pushed the price up or down for that specific house.

**Exercise 3**: Extend the FastAPI with a `/predict_batch` endpoint that accepts CSV files, processes them asynchronously (using BackgroundTasks), and returns a job ID for status checking.

### Hard
**Exercise 4**: Implement A/B testing logic in the API. Maintain two model versions (v1.0 and v1.1) and route traffic between them based on a header or query parameter. Log which version served each request for comparison.

### Bonus
**Exercise 5**: Containerize the entire application with Docker. Create:
- A `Dockerfile` for the FastAPI service
- A `docker-compose.yml` that runs both the API and Streamlit UI
- Environment variable configuration for model paths and ports

Push to a container registry and deploy to a cloud platform (AWS/GCP/Azure).

<details>
<summary>📋 Exercise Solutions & Extension Ideas (Click to expand)</summary>

### Exercise 1: Titanic Dataset

```python
from sklearn.datasets import fetch_openml
from sklearn.metrics import roc_auc_score, f1_score

# Load Titanic
titanic = fetch_openml('titanic', version=1, as_frame=True, parser='auto')
X, y = titanic.data, titanic.target

# Convert target to binary
y = (y == '1').astype(int)

# Key differences for classification:
# - Use StratifiedKFold for CV
# - Metrics: roc_auc, f1, precision, recall
# - Models: XGBClassifier, LGBMClassifier, CatBoostClassifier
# - Target encoding needs regularization (smoothing)
# - Handle missing ages with iterative imputer
```

### Exercise 2: SHAP in Streamlit

```python
import streamlit as st
import shap

# After prediction in app.py:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_proc)

# Display force plot
st.subheader("Prediction Explanation")
fig, ax = plt.subplots()
shap.force_plot(
    explainer.expected_value, 
    shap_values[0], 
    input_data.iloc[0],
    matplotlib=True,
    show=False
)
st.pyplot(fig)
```

### Exercise 3: Async Batch Processing

```python
from fastapi import BackgroundTasks
import uuid

jobs = {}

@app.post("/predict_batch_async")
def predict_batch_async(file: UploadFile, background_tasks: BackgroundTasks):
    job_id = str(uuid.uuid4())
    jobs[job_id] = {"status": "processing", "result": None}
    
    background_tasks.add_task(process_batch, job_id, file)
    return {"job_id": job_id, "status": "processing"}

@app.get("/job/{job_id}")
def get_job_status(job_id: str):
    return jobs.get(job_id, {"error": "Job not found"})

def process_batch(job_id, file):
    # Read CSV, process, predict, save to storage
    # Update jobs[job_id] = {"status": "complete", "result": ...}
    pass
```

### Exercise 4: A/B Testing

```python
@app.post("/predict")
def predict(house: HouseFeatures, model_version: str = "v1"):
    if model_version == "v1":
        model = model_v1
    elif model_version == "v2":
        model = model_v2
    else:
        raise HTTPException(400, "Invalid version")
    
    # Log which version was used
    logger.info(f"Prediction using {model_version}", extra={"version": model_version})
    
    # ... predict and return
```

### Exercise 5: Docker Setup

**Dockerfile:**
```dockerfile
FROM python:3.9-slim

WORKDIR /app
COPY requirements.txt .
RUN pip install -r requirements.txt

COPY capstone_model_v1.joblib .
COPY api.py .

CMD ["uvicorn", "api:app", "--host", "0.0.0.0", "--port", "8000"]
```

**docker-compose.yml:**
```yaml
version: '3.8'
services:
  api:
    build: .
    ports:
      - "8000:8000"
    volumes:
      - ./models:/app/models
  
  ui:
    build: .
    command: streamlit run app.py --server.port 8501
    ports:
      - "8501:8501"
    depends_on:
      - api
```

</details>

## 🎓 Summary – The Full ML Mastery Journey

Congratulations! You've completed the 30-notebook ML Mastery Series for Python Experts. Let's recap what you've accomplished:

**Series Highlights:**
- **Notebooks 1-5**: Foundations – Python, NumPy, Pandas, Visualization, and statistical thinking
- **Notebooks 6-10**: Core ML – Scikit-learn, preprocessing, linear models, trees, and model evaluation
- **Notebooks 11-15**: Advanced Algorithms – SVMs, ensembles, gradient boosting, and hyperparameter tuning
- **Notebooks 16-20**: Specialized Techniques – Time series, NLP, clustering, dimensionality reduction, and anomaly detection
- **Notebooks 21-25**: Production & XAI – Pipelines, model serialization, SHAP, LIME, and MLOps fundamentals
- **Notebooks 26-30**: Cutting Edge – Deep learning, Transformers, Graph ML, AutoML, and this final capstone

**In this capstone, you demonstrated:**
- End-to-end data science workflow from exploration to deployment
- Advanced feature engineering with custom sklearn transformers
- State-of-the-art gradient boosting with XGBoost, LightGBM, and CatBoost
- Rigorous evaluation with hold-out testing and residual analysis
- Model interpretability using SHAP and partial dependence plots
- Production deployment via FastAPI REST services
- Interactive applications with Streamlit for business users
- Best practices for reproducibility, versioning, and monitoring

**The skills you've gained:**
✅ Building robust, production-grade ML pipelines  
✅ Selecting and tuning algorithms for specific problem types  
✅ Explaining model decisions to stakeholders  
✅ Deploying models as scalable services  
✅ Engineering features that generalize  
✅ Avoiding data leakage and overfitting  
✅ Debugging model failures systematically  

---

## 🚀 Congratulations!

**You've completed the 30-notebook ML Mastery Series for Python Experts. Go build something amazing.**

The journey doesn't end here—keep experimenting, stay curious, and remember: the best way to master machine learning is to ship models that solve real problems. Whether you're building recommendation systems, fraud detection, computer vision apps, or NLP services, you now have the complete toolkit to take ideas from concept to production.

**Happy modeling! 🤖✨**